In [1]:
import pandas as pd
import numpy as np
import torch
import random
import os

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
    DataCollatorWithPadding
)

from peft import get_peft_model, LoraConfig, TaskType

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_recall_fscore_support,
    matthews_corrcoef, balanced_accuracy_score,
    cohen_kappa_score, jaccard_score, hamming_loss,
    confusion_matrix
)


/usr/local/lib/python3.10/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/usr/local/lib/python3.10/dist-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
2025-06-20 07:12:56.637592: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-20 07:12:57.496800: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
202

In [3]:
import wandb
import optuna
import huggingface_hub

wandb.login(key="WANDB_KEY")
wandb.init(project="llama2_degendered", reinit=True)

huggingface_hub.login(token="HUGGINGFACE_TOKEN")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/hice1/mwesley32/.netrc


In [4]:
model_name = "meta-llama/Llama-2-7b-hf"
peft_cache_path = "../scratch/cache/llama2_degendered_hpo"

target_modules = ["q_proj", "v_proj", "k_proj", "o_proj"]

In [5]:
# Ensure 'data/combined_letters_degendered.csv' exists at the specified path
df = pd.read_csv("data/combined_letters_degendered.csv")[["full_text", "label"]].dropna()
df["label"] = df["label"].astype(int)

# Perform train-test split, stratifying by label to maintain class distribution
X_train, X_test, y_train, y_test = train_test_split(
    df["full_text"],
    df["label"],
    test_size=0.2,
    stratify=df["label"],
)

# The tokenizer is always loaded from the base model, as we need a fresh tokenizer for each run
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

In [6]:
# This function prepares your text data for the model by converting it into token IDs
from datasets import Dataset
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding=False,
        max_length=512
    )
    tokens["labels"] = example["label"]
    return tokens

# Convert pandas Series to Hugging Face Dataset objects, then map the tokenization function
train_dataset = Dataset.from_dict({"text": X_train.tolist(), "label": y_train.tolist()})
test_dataset = Dataset.from_dict({"text": X_test.tolist(), "label": y_test.tolist()})

tokenized_train = train_dataset.map(tokenize, batched=True, num_proc=2).remove_columns(["text"])
tokenized_test = test_dataset.map(tokenize, batched=True, num_proc=2).remove_columns(["text"])

# Data Collator: Dynamically pads input sequences to the longest sequence in the batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map (num_proc=2):   0%|          | 0/7189 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/1798 [00:00<?, ? examples/s]

In [7]:
# Compute metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    preds = logits.argmax(-1)
    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)
    
    print("Confusion Matrix:\n", confusion_matrix(labels, preds))

    return {
        "f1_score": f1, 
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "mcc": matthews_corrcoef(labels, preds),
        "balanced_accuracy": balanced_accuracy_score(labels, preds),
        "cohen_kappa": cohen_kappa_score(labels, preds),
        "jaccard": jaccard_score(labels, preds, average="macro"),
        "hamming_loss": hamming_loss(labels, preds)
    }


In [8]:
def model_init_for_hpo(trial=None): # Make 'trial' parameter optional, defaulting to None
    # Load the base pre-trained model (Llama-2-7b-hf)
    base_model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        id2label={0: "female", 1: "male"},
        label2id={"female": 0, "male": 1},
        cache_dir=peft_cache_path,
        device_map="auto"
    )
    base_model.config.pad_token_id = tokenizer.pad_token_id

    # Define LoRA hyperparameters if a trial is active,
    # otherwise fall back to a default/baseline configuration.
    if trial:
        lora_r = trial.suggest_int("lora_r", 8, 32, step=8)
        lora_alpha = trial.suggest_int("lora_alpha", lora_r * 2, lora_r * 4, step=lora_r)
        lora_dropout = trial.suggest_float("lora_dropout", 0.0, 0.2, step=0.05)
    else:
        lora_r = 8
        lora_alpha = 16
        lora_dropout = 0.1

    # Create the LoRA configuration for the current trial
    current_lora_config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        bias="none",
        task_type=TaskType.SEQ_CLS,
        target_modules=target_modules
    )

    # Apply PEFT (LoRA) to the base model
    peft_model = get_peft_model(base_model, current_lora_config)

    return peft_model

In [9]:
def optuna_hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True),
        "num_train_epochs": trial.suggest_int("num_train_epochs", 3, 10),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [2, 4, 8, 16]),
        "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.03, step=0.01),
    }


In [10]:
training_args_for_hpo = TrainingArguments(
    output_dir="../scratch/hpo_results_llama2_degendered",
    per_device_eval_batch_size=8,
    fp16=True,
    save_strategy="steps",
    save_steps=500,
    logging_steps=50,
    report_to="wandb",
    remove_unused_columns=False,
    load_best_model_at_end=False,
    eval_strategy="epoch",
)

trainer = Trainer(
    model_init=model_init_for_hpo,
    args=training_args_for_hpo,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


/tmp/ipykernel_1655097/916219255.py:14: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


config.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at meta-llama/Llama-2-7b-hf and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [ ]:
best_run = trainer.hyperparameter_search(
    direction="maximize",
    backend="optuna",
    hp_space=optuna_hp_space,
    n_trials=30,
    storage="sqlite:///../scratch/optuna_llama2_gendered.db",
    study_name="llama2_degender_hpo",
    load_if_exists=True,
    compute_objective=lambda metrics: metrics["eval_f1_score"]
)

print("Best run details:")
print(best_run)

# Access the best hyperparameters found by Optuna
best_hps = best_run.hyperparameters
print("\nBest Hyperparameters Found:")
for hp, value in best_hps.items():
    print(f"  {hp}: {value}")

# W&B will provide a URL to the best run in its logs.
print(f"\nFind the best run and explore all trials in W&B at: {best_run.url}")

[I 2025-06-20 07:37:38,785] A new study created in RDB with name: llama2_degender_hpo


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at meta-llama/Llama-2-7b-hf and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,F1 Score,Accuracy,Precision,Recall,Mcc,Balanced Accuracy,Cohen Kappa,Jaccard,Hamming Loss,Runtime,Samples Per Second,Steps Per Second
1,0.000000,nan,0.146541,0.309789,0.095969,0.309789,0.000000,0.500000,0.000000,0.154894,0.690211,36.323500,49.500000,6.194000
2,0.000000,nan,0.146541,0.309789,0.095969,0.309789,0.000000,0.500000,0.000000,0.154894,0.690211,36.292100,49.542000,6.200000
3,0.000000,nan,0.146541,0.309789,0.095969,0.309789,0.000000,0.500000,0.000000,0.154894,0.690211,36.279600,49.560000,6.202000


Confusion Matrix:
 [[ 557    0]
 [1241    0]]
Confusion Matrix:
 [[ 557    0]
 [1241    0]]
Confusion Matrix:
 [[ 557    0]
 [1241    0]]


...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at meta-llama/Llama-2-7b-hf and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


eval/accuracy,▁▇██▆
eval/balanced_accuracy,▁▆▅▅█
eval/cohen_kappa,▁▆▅▆█
eval/f1_score,▁▇▆▆█
eval/hamming_loss,█▂▁▁▃
eval/jaccard,▁▇▅▆█
eval/loss,▁▁▂▇█
eval/mcc,▁▇▆▇█
eval/precision,▁▇▇██
eval/recall,▁▇██▆
eval/runtime,▁▁▄██


Epoch,Training Loss,Validation Loss,F1 Score,Accuracy,Precision,Recall,Mcc,Balanced Accuracy,Cohen Kappa,Jaccard,Hamming Loss,Runtime,Samples Per Second,Steps Per Second
1,0.812300,0.743670,0.592421,0.657953,0.582884,0.657953,0.019563,0.506318,0.015551,0.367517,0.342047,36.759700,48.912000,6.121000
2,0.699500,0.691512,0.605065,0.660734,0.598309,0.660734,0.050955,0.517733,0.042791,0.379901,0.339266,36.736600,48.943000,6.125000


Confusion Matrix:
 [[  60  497]
 [ 118 1123]]
Confusion Matrix:
 [[  79  478]
 [ 132 1109]]


In [ ]:
# Re-training the model with the best hyperparameters
best_hps = best_run.hyperparameters

# Re-initialize the base model
final_base_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "female", 1: "male"},
    label2id={"female": 0, "male": 1},
    cache_dir=peft_cache_path,
    device_map="auto"
)

final_base_model.config.pad_token_id = tokenizer.pad_token_id

# Re-create the LoRA configuration using the best 'r', 'lora_alpha', 'lora_dropout'
final_lora_config = LoraConfig(
    r=best_hps["lora_r"],
    lora_alpha=best_hps["lora_alpha"],
    lora_dropout=best_hps["lora_dropout"],
    bias="none",
    task_type=TaskType.SEQ_CLS,
    target_modules=target_modules
)

# Apply PEFT to the final base model
final_peft_model = get_peft_model(final_base_model, final_lora_config)
final_peft_model.print_trainable_parameters()

# Define TrainingArguments for the final training run
final_model_output_dir = "../scratch/final_llama2_degendered_model"
final_training_args = TrainingArguments(
    output_dir=final_model_output_dir,
    per_device_train_batch_size=best_hps["per_device_train_batch_size"],
    per_device_eval_batch_size=best_hps["per_device_train_batch_size"],
    num_train_epochs=best_hps["num_train_epochs"],
    learning_rate=best_hps["learning_rate"],
    weight_decay=best_hps["weight_decay"],
    fp16=True,
    save_strategy="epoch",
    logging_steps=50,
    report_to="wandb",
    remove_unused_columns=False,
    load_best_model_at_end=True,
    metric_for_best_model="f1_score",
    evaluation_strategy="epoch",
    save_total_limit=1,
    run_name="final_model_training_best_hps"
)

# Initialize the Trainer for final training
final_trainer = Trainer(
    model=final_peft_model,
    args=final_training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# Train the final model
final_trainer.train()

# Evaluate the final (best) model
eval_results = final_trainer.evaluate()
print("\nFinal Model Evaluation Results:")
print(eval_results)

# Save the final trained PEFT model and tokenizer
final_trainer.save_model(final_model_output_dir)
tokenizer.save_pretrained(final_model_output_dir)

print(f"\nFinal model and tokenizer saved to: {final_model_output_dir}")
